## Pitch Visualization

In [262]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Datenset laden

In [263]:
data = pd.read_csv('../data/raw/DATASET CSV 11-02-2026.csv', sep=';')


# Überblick über die Daten schaffen

In [264]:
data.head()
data.shape
data.columns
data.info()
data.isna().sum()
unique_titel = data['titel_kurz_d'].unique()
len(unique_titel)
unique_titel[:20]

<class 'pandas.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Columns: 874 entries, anr to nach_cockpit_e
dtypes: float64(480), int64(10), str(384)
memory usage: 4.7 MB


<StringArray>
[                     'Bundesverfassung der schweizerischen Eidgenossenschaft',
                                                            'Mass und Gewicht',
    'Gleichstellung der Juden und Naturalisierten mit Bezug auf Niederlassung',
                  'Stimmrecht der Niedergelassenen in Gemeindeangelegenheiten',
           'Besteuerung und zivilrechtliche Verhältnisse der Niedergelassenen',
               'Stimmrecht der Niedergelassenen in kantonalen Angelegenheiten',
                                                'Glaubens- und Kultusfreiheit',
                                         'Ausschliessung einzelner Strafarten',
                                              'Schutz des geistigen Eigentums',
                                        'Verbot der Lotterie und Hasardspiele',
                                            'Bundesverfassung (Totalrevision)',
 'Gesetz betreffend Feststellung und Beurkundung des Zivilstandes und die Ehe',
            'Gesetz über d

# Wrangling und Datacleaning

## Datatypes

In [265]:
data['datum'] = pd.to_datetime(data['datum'], format='%d.%m.%Y')
data['jahr'] = data['datum'].dt.year



numeric_cols = ["d1e1", "d1e2", "d1e3", "br-pos", "annahme",
                "volkja-proz", "berecht", "stimmen", "gultig",
                "leer", "rechtsform"]

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

data["bet"] = data["bet"].replace(r"^\s*\.\s*$", np.nan, regex=True)
data["beteiligung"] = pd.to_numeric(data["bet"], errors='coerce')

parteien = [
    'p-fdp', 'p-sps', 'p-svp', 'p-mitte', 'p-evp', 'p-gps', 'p-glp',
    'p-ucsp', 'p-pda', 'p-sd', 'p-edu', 'p-fps', 'p-lega', 'p-kvp',
    'p-mcg', 'p-cvp', 'p-bdp', 'p-lps', 'p-ldu', 'p-poch', 'p-rep',
    'p-eco', 'p-sgv', 'p-sbv', 'p-sgb', 'p-travs', 'p-sav', 'p-vsa',
    'p-vpod', 'p-voev', 'p-tcs', 'p-vcs', 'p-acs', 'p-sbk', 'p-ssv',
    'p-gem', 'p-kdk', 'p-kkjpd', 'p-gdk', 'p-ldk', 'p-vdk', 'p-sodk',
    'p-endk', 'p-fdk', 'p-edk', 'p-bpuk'
]

for col in parteien:
    if col in data.columns:
        data[col] = data[col].replace(r"^\s*\.\s*$", np.nan, regex=True)
        data[col] = data[col].replace(9999, np.nan)
        data[col] = pd.to_numeric(data[col], errors='coerce')

data.replace([9999,999], np.nan, inplace=True)

p-fdp
1.0       390
2.0       256
9999.0     39
5.0         8
3.0         7
NaN         5
8.0         3
Name: count, dtype: int64
float64
0      9999.0
1      9999.0
2      9999.0
3      9999.0
4      9999.0
        ...  
703       2.0
704       2.0
705       1.0
706       NaN
707       NaN
Name: p-fdp, Length: 708, dtype: float64


/var/folders/pp/lkq7hgk93553k7jf082bnq1r0000gn/T/ipykernel_68325/2660446436.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['jahr'] = data['datum'].dt.year
/var/folders/pp/lkq7hgk93553k7jf082bnq1r0000gn/T/ipykernel_68325/2660446436.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["beteiligung"] = pd.to_numeric(data["bet"], errors='coerce')


,anr,datum,titel_kurz_d,titel_kurz_f,titel_kurz_e,titel_off_d,titel_off_f,stichwort,swissvoteslink,anzahl,...,bfsdash-fr,bfsdash-en,bfsmap-de,bfsmap-fr,bfsmap-en,nach_cockpit_d,nach_cockpit_f,nach_cockpit_e,jahr,beteiligung
0,1.0,1848-09-12,Bundesverfassung der schweizerischen Eidgenoss...,Constitution fédérale de la Confédération suisse,Federal Constitution of the Swiss Confederation,Totalrevision vom 12. September 1848,Révision totale du 12 septembre 1848,.,https://swissvotes.ch/vote/1.00,1,...,.,.,NaN,NaN,.,.,.,.,1848,NaN
1,2.0,1866-01-14,Mass und Gewicht,Poids et mesures,Weights and measures,Festsetzung von Mass und Gewicht,Poids et mesures,.,https://swissvotes.ch/vote/2.00,9,...,.,.,https://mapexplorer.bfs.admin.ch/?obs=politic&...,https://mapexplorer.bfs.admin.ch/?obs=politic&...,.,.,.,.,1866,NaN
2,3.0,1866-01-14,Gleichstellung der Juden und Naturalisierten m...,Egalité des Juifs,Equal rights for Jews and naturalised citizens...,Gleichstellung der Juden und Naturalisierten m...,Egalité des citoyens au point de vue de l'étab...,.,https://swissvotes.ch/vote/3.00,9,...,.,.,https://mapexplorer.bfs.admin.ch/?obs=politic&...,https://mapexplorer.bfs.admin.ch/?obs=politic&...,.,.,.,.,1866,NaN
3,4.0,1866-01-14,Stimmrecht der Niedergelassenen in Gemeindeang...,"Droit de vote des Suisses établis, en matière ...",Swiss residents' right to vote in communal mat...,Stimmrecht der Niedergelassenen in Gemeindeang...,"Droit de vote des Suisses établis, en matière ...",.,https://swissvotes.ch/vote/4.00,9,...,.,.,https://mapexplorer.bfs.admin.ch/?obs=politic&...,https://mapexplorer.bfs.admin.ch/?obs=politic&...,.,.,.,.,1866,NaN
4,5.0,1866-01-14,Besteuerung und zivilrechtliche Verhältnisse d...,Impôts et rapports civils des Suisses établis,Taxation and civil law status of Swiss residents,Besteuerung und zivilrechtliche Verhältnisse d...,Impôts et rapports civils des Suisses établis,.,https://swissvotes.ch/vote/5.00,9,...,.,.,https://mapexplorer.bfs.admin.ch/?obs=politic&...,https://mapexplorer.bfs.admin.ch/?obs=politic&...,.,.,.,.,1866,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
703,683.0,2026-03-08,SRG-Initiative «200 Franken sind genug!»,"Initiative SSR « 200 francs, ça suffit ! »",Initiative «200 francs is enough!» (fees for p...,Volksinitiative «200 Franken sind genug! (SRG-...,"Initiative populaire « 200 francs, ça suffit !...",«Halbierungsinitiative»,https://swissvotes.ch/vote/683.00,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN
704,684.0,2026-03-08,Klimafonds-Initiative,Initiative pour un fonds climat,Climate Fund Initiative,Volksinitiative «Für eine gerechte Energie- un...,Initiative populaire « Pour une politique éner...,.,https://swissvotes.ch/vote/684.00,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN
705,685.0,2026-03-08,Gesetz über die Individualbesteuerung,Loi sur l'imposition individuelle,Act on Individual Taxation,Bundesgesetz über die Individualbesteuerung,Loi fédérale sur l’imposition individuelle,.,https://swissvotes.ch/vote/685.00,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN
706,686.0,2026-06-14,«Nachhaltigkeitsinitiative»,« Initiative pour la durabilité »,«Sustainability initiative» (Limiting the nati...,Volksinitiative «Keine 10-Millionen-Schweiz! (...,Initiative populaire « Pas de Suisse à 10 mill...,.,https://swissvotes.ch/vote/686.00,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN


## Mapping

### Thematische Gruppen der Abstimmungen
Fokus auf Einteilung durch Swissvote (in Abstimmung mit BFS), da es sich um eine inhaltliche Überarbeitung der ursprünglichen BFS Position geht. Deshalb werden d2 und d3 nicht berücksichtigt.

In [266]:
data.head()

,anr,datum,titel_kurz_d,titel_kurz_f,titel_kurz_e,titel_off_d,titel_off_f,stichwort,swissvoteslink,anzahl,...,bfsdash-fr,bfsdash-en,bfsmap-de,bfsmap-fr,bfsmap-en,nach_cockpit_d,nach_cockpit_f,nach_cockpit_e,jahr,beteiligung
0,1.0,1848-09-12,Bundesverfassung der schweizerischen Eidgenoss...,Constitution fédérale de la Confédération suisse,Federal Constitution of the Swiss Confederation,Totalrevision vom 12. September 1848,Révision totale du 12 septembre 1848,.,https://swissvotes.ch/vote/1.00,1,...,.,.,NaN,NaN,.,.,.,.,1848,NaN
1,2.0,1866-01-14,Mass und Gewicht,Poids et mesures,Weights and measures,Festsetzung von Mass und Gewicht,Poids et mesures,.,https://swissvotes.ch/vote/2.00,9,...,.,.,https://mapexplorer.bfs.admin.ch/?obs=politic&...,https://mapexplorer.bfs.admin.ch/?obs=politic&...,.,.,.,.,1866,NaN
2,3.0,1866-01-14,Gleichstellung der Juden und Naturalisierten m...,Egalité des Juifs,Equal rights for Jews and naturalised citizens...,Gleichstellung der Juden und Naturalisierten m...,Egalité des citoyens au point de vue de l'étab...,.,https://swissvotes.ch/vote/3.00,9,...,.,.,https://mapexplorer.bfs.admin.ch/?obs=politic&...,https://mapexplorer.bfs.admin.ch/?obs=politic&...,.,.,.,.,1866,NaN
3,4.0,1866-01-14,Stimmrecht der Niedergelassenen in Gemeindeang...,"Droit de vote des Suisses établis, en matière ...",Swiss residents' right to vote in communal mat...,Stimmrecht der Niedergelassenen in Gemeindeang...,"Droit de vote des Suisses établis, en matière ...",.,https://swissvotes.ch/vote/4.00,9,...,.,.,https://mapexplorer.bfs.admin.ch/?obs=politic&...,https://mapexplorer.bfs.admin.ch/?obs=politic&...,.,.,.,.,1866,NaN
4,5.0,1866-01-14,Besteuerung und zivilrechtliche Verhältnisse d...,Impôts et rapports civils des Suisses établis,Taxation and civil law status of Swiss residents,Besteuerung und zivilrechtliche Verhältnisse d...,Impôts et rapports civils des Suisses établis,.,https://swissvotes.ch/vote/5.00,9,...,.,.,https://mapexplorer.bfs.admin.ch/?obs=politic&...,https://mapexplorer.bfs.admin.ch/?obs=politic&...,.,.,.,.,1866,NaN


In [267]:
# Hauptgruppe (d1e1)
def assign_hauptgruppe(row):
    d1 = row['d1e1']
    mapping = {
        1: 'Staatsordnung',
        2: 'Aussenpolitik',
        3: 'Sicherheitspolitik',
        4: 'Wirtschaft',
        5: 'Landwirtschaft',
        6: 'Öffentliche Finanzen',
        7: 'Energie',
        8: 'Verkehr & Infrastruktur',
        9: 'Umwelt & Lebensraum',
        10: 'Sozial- und Gesellschaftspolitik',
        11: 'Bildung & Forschung',
        12: 'Kultur, Religion & Medien',
    }
    return mapping.get(d1, 'Andere')

def assign_untergruppen(row):
    d2 = row['d1e2']

    mapping = {
        1.1: 'Nationale Identität',
        1.2: 'Politisches System',
        1.3: 'Institutionen',
        1.4: 'Volksrechte',
        1.5: 'Föderalismus',
        1.6: 'Rechtsordnung',
        2.1: 'Aussenpolitische Grundhaltung',
        2.2: 'Europapolitik',
        2.3: 'Internationale Organisationen',
        2.4: 'Entwicklungszusammenarbeit',
        2.5: 'Staatsverträge',
        2.6: 'Aussenwirtschaftspolitik',
        2.7: 'Diplomatie',
        2.8: 'Auslandschweizer:innen',
        3.1: 'Öffentliche Sicherheit',
        3.2: 'Armee',
        3.3: 'Landesversorgung',
        4.1: 'Wirtschaftspolitik',
        4.2: 'Arbeit und Beschäftigung',
        4.3: 'Finanzwesen',
        4.4: 'Freizeit und Tourismus',
        5.1: 'Agrarpolitik',
        5.2: 'Tierische Produktion',
        5.3: 'Pflanzliche Produktion',
        5.4: 'Forstwirtschaft',
        5.5: 'Fischerei, Jagd, Haustiere',
        6.1: 'Steuerwesen',
        6.2: 'Finanzordnung',
        6.3: 'Öffentliche Ausgaben',
        6.4: 'Spar- und Sanierungsmassnahmen',
        7.1: 'Energiepolitik',
        7.2: 'Kernenergie',
        7.3: 'Wasserkraft',
        7.4: 'Alternativenergien',
        7.5: 'Erdöl, Gas',
        8.1: 'Verkehrspolitik',
        8.2: 'Strassenverkehr',
        8.3: 'Schienenverkehr',
        8.4: 'Luftverkehr',
        8.5: 'Schifffahrt',
        8.6: 'Post',
        8.7: 'Telekommunikation',
        9.1: 'Boden',
        9.2: 'Wohnen',
        9.3: 'Umwelt',
        10.1: 'Gesundheit',
        10.2: 'Sozialversicherungen',
        10.3: 'Gesellschaftsfragen',
        11.1: 'Bildungspolitik',
        11.2: 'Schulen',
        11.3: 'Hochschulen',
        11.4: 'Forschung',
        11.5: 'Berufsbildung',
        12.1: 'Kulturpolitik',
        12.2: 'Sprachpolitik',
        12.3: 'Religion, Kirchen',
        12.4: 'Sport',
        12.5: 'Medien und Kommunikation',
    }
    return mapping.get(d2, 'Andere')

# Kleinstgruppe (d1e3)
def assign_kleinstgruppe(row):
    d3 = row['d1e3']
    mapping = {
        1.21: 'Bundesverfassung',
        1.22: 'Verfassungsgebungsverfahren',
        1.23: 'Gesetzgebungsverfahren',
        1.24: 'Wahlsystem',
        1.31: 'Regierung, Verwaltung',
        1.32: 'Parlament',
        1.33: 'Gerichte',
        1.34: 'Nationalbank',
        1.41: 'Initiative',
        1.42: 'Referendum',
        1.43: 'Stimmrecht',
        1.51: 'Territorialfragen',
        1.52: 'Beziehungen Bund–Kantone',
        1.53: 'Aufgabenteilung',
        1.61: 'Internationales Recht',
        1.62: 'Grundrechte',
        1.63: 'Bürgerrecht',
        1.64: 'Privatrecht',
        1.65: 'Strafrecht',
        1.66: 'Datenschutz',
        2.11: 'Neutralität',
        2.12: 'Unabhängigkeit',
        2.13: 'Gute Dienste',
        2.21: 'EFTA',
        2.22: 'EU',
        2.23: 'EWR',
        2.24: 'Andere europäische Organisationen',
        2.31: 'UNO',
        2.32: 'Andere internationale Organisationen',
        2.61: 'Exportförderung',
        2.62: 'Zollwesen',
        3.11: 'Bevölkerungsschutz',
        3.12: 'Staatsschutz',
        3.13: 'Polizei',
        3.21: 'Armee (allgemein)',
        3.22: 'Militärorganisation',
        3.23: 'Rüstung',
        3.24: 'Militäranlagen',
        3.25: 'Dienstverweigerung, Zivildienst',
        3.26: 'Armeeabschaffung',
        3.27: 'Militärische Ausbildung',
        3.28: 'Internationale Einsätze',
        4.11: 'Konjunkturpolitik',
        4.12: 'Wettbewerbspolitik',
        4.13: 'Strukturpolitik',
        4.14: 'Preispolitik',
        4.15: 'Konsumentenschutz',
        4.16: 'Gesellschaftsrecht',
        4.21: 'Arbeitsbedingungen',
        4.22: 'Arbeitszeit',
        4.23: 'Sozialpartnerschaft',
        4.24: 'Beschäftigungspolitik',
        4.31: 'Geld- und Währungspolitik',
        4.32: 'Banken, Börsen, Versicherungen',
        4.41: 'Fremdenverkehr',
        4.42: 'Hotellerie und Gastgewerbe',
        4.43: 'Geldspiele',
        6.11: 'Steuerpolitik',
        6.12: 'Steuersystem',
        6.13: 'Direkte Steuern',
        6.14: 'Indirekte Steuern',
        8.11: 'Agglomerationsverkehr',
        8.12: 'Transitverkehr',
        8.21: 'Strassenbau',
        8.22: 'Schwerverkehr',
        8.31: 'Güterverkehr',
        8.32: 'Personenverkehr',
        9.11: 'Raumplanung',
        9.12: 'Bodenrecht',
        9.21: 'Mietwesen',
        9.22: 'Wohnungsbau, Wohneigentum',
        9.31: 'Umweltpolitik',
        9.32: 'Lärmschutz',
        9.33: 'Luftreinhaltung',
        9.34: 'Gewässerschutz',
        9.35: 'Bodenschutz',
        9.36: 'Abfälle',
        9.37: 'Natur- und Heimatschutz',
        9.38: 'Tierschutz',
        10.11: 'Gesundheitspolitik',
        10.12: 'Medizinforschung und -technik',
        10.13: 'Medikamente',
        10.14: 'Suchtmittel',
        10.15: 'Fortpflanzungsmedizin',
        10.21: 'AHV',
        10.22: 'Invalidenversicherung',
        10.23: 'Berufliche Vorsorge',
        10.24: 'Kranken- und Unfallversicherung',
        10.25: 'Mutterschaftsversicherung',
        10.26: 'Arbeitslosenversicherung',
        10.27: 'Erwerbsersatzordnung',
        10.28: 'Fürsorge',
        10.31: 'Migrations- und Integrationspolitik',
        10.32: 'Asylpolitik',
        10.33: 'Frauen und Gleichstellungspolitik',
        10.34: 'Familienpolitik',
        10.35: 'Kinder- und Jugendpolitik',
        10.36: 'Alterspolitik',
        10.37: 'Menschen mit Behinderungen',
        10.38: 'LGBTQIA+',
        11.41: 'Gentechnologie',
        11.42: 'Tierversuche',
        12.51: 'Medienpolitik',
        12.52: 'Presse',
        12.53: 'Radio, Fernsehen, Elektronische Medien',
        12.54: 'Medienfreiheit',
    }
    return mapping.get(d3, None)


# Anwendung
data['hauptgruppe'] = data.apply(assign_hauptgruppe, axis=1)
data['untergruppe'] = data.apply(assign_untergruppen, axis=1)
data['kleinstgruppe'] = data.apply(assign_kleinstgruppe, axis=1)


print("N hauptgruppe", data['hauptgruppe'].value_counts(dropna=False))
print("N untergruppe", data['untergruppe'].value_counts(dropna=False))
print("N kleinstgruppe", data['kleinstgruppe'].value_counts(dropna=False))

N hauptgruppe hauptgruppe
Sozial- und Gesellschaftspolitik    142
Staatsordnung                       107
Wirtschaft                           86
Öffentliche Finanzen                 71
Verkehr & Infrastruktur              58
Sicherheitspolitik                   53
Umwelt & Lebensraum                  49
Landwirtschaft                       38
Aussenpolitik                        34
Bildung & Forschung                  25
Energie                              24
Kultur, Religion & Medien            21
Name: count, dtype: int64
N untergruppe untergruppe
Sozialversicherungen              61
Wirtschaftspolitik                50
Steuerwesen                       42
Gesellschaftsfragen               42
Armee                             40
Gesundheit                        39
Rechtsordnung                     35
Institutionen                     32
Strassenverkehr                   29
Volksrechte                       21
Arbeit und Beschäftigung          19
Umwelt                            1

In [268]:
data[['titel_kurz_d', 'd1e1', 'd1e2', 'd1e3', 'hauptgruppe', 'untergruppe', 'kleinstgruppe']].sample(20, random_state=42)


,titel_kurz_d,d1e1,d1e2,d1e3,hauptgruppe,untergruppe,kleinstgruppe
120,Neuordnung der militärischen Ausbildung,3,3.2,3.27,Sicherheitspolitik,Armee,Militärische Ausbildung
247,Tierschutzartikel,9,9.3,9.38,Umwelt & Lebensraum,Umwelt,Tierschutz
324,Energieartikel,7,7.1,NaN,Energie,Energiepolitik,NaN
204,Natur- und Heimatschutz-Artikel,9,9.3,9.37,Umwelt & Lebensraum,Umwelt,Natur- und Heimatschutz
603,Initiative «Schluss mit der MwSt-Diskriminieru...,6,6.1,6.14,Öffentliche Finanzen,Steuerwesen,Indirekte Steuern
356,Asylgesetz,10,10.3,10.32,Sozial- und Gesellschaftspolitik,Gesellschaftsfragen,Asylpolitik
81,Initiative für ein Verbot der Spielbanken,4,4.1,4.13,Wirtschaft,Wirtschaftspolitik,Strukturpolitik
54,Vereinheitlichung des Strafrechts,1,1.6,1.65,Staatsordnung,Rechtsordnung,Strafrecht
512,Gesetz über den Zivilschutz,3,3.1,3.11,Sicherheitspolitik,Öffentliche Sicherheit,Bevölkerungsschutz
572,Initiative «Für den Schutz vor Waffengewalt»,1,1.6,NaN,Staatsordnung,Rechtsordnung,NaN


### Rechtsform

In [269]:
rechtsform_map = {
    1: 'Obligatorisches Referendum',
    2: 'Fakultatives Referendum',
    3: 'Volksinitiative',
    4: 'Direkter Gegenentwurf',
    5: 'Stichfrage'
}

data['rechtsform_name'] = data['rechtsform'].map(rechtsform_map)
print(data['rechtsform_name'])

0      Obligatorisches Referendum
1      Obligatorisches Referendum
2      Obligatorisches Referendum
3      Obligatorisches Referendum
4      Obligatorisches Referendum
                  ...            
703               Volksinitiative
704               Volksinitiative
705       Fakultatives Referendum
706               Volksinitiative
707       Fakultatives Referendum
Name: rechtsform_name, Length: 708, dtype: str


### Kantonsangaben
In alten Notebooks nachvollziehbar, noch nicht klar ob wir es brauchen

### Ev. Als Kontrolle die Wählendenanteile bei den letzten nationalen Wahlen

In [270]:
x = ['w-fdp', 'w-sps', 'w-svp', 'w-mitte', 'w-evp', 'w-gps', 'w-glp', 'w-ucsp', 'w-pda', 'w-sd', 'w-edu', 'w-fps', 'w-lega', 'w-kvp', 'w-mcg', 'w-cvp', 'w-bdp', 'w-lps', 'w-ldu', 'w-poch', 'w-rep','w-ubrige']

# Neuer DF in einem separaten CSV erstellen

In [271]:
df = data[['anr', 'datum', 'legisjahr', 'rechtsform_name', 'titel_kurz_d', 'anzahl','beteiligung', 'annahme', 'volkja-proz', 'berecht', 'stimmen', 'gultig', 'leer', 'hauptgruppe', 'untergruppe', 'kleinstgruppe', 'br-pos', 'sr-pos', 'nr-pos', 'bv-pos', 'srja', 'srnein', 'nrja', 'nrnein',  'sammeltempo','p-fdp', 'p-sps', 'p-svp', 'p-mitte', 'p-evp', 'p-gps', 'p-glp', 'p-ucsp', 'p-pda', 'p-sd', 'p-edu', 'p-fps', 'p-lega', 'p-kvp', 'p-mcg', 'p-cvp', 'p-bdp', 'p-lps', 'p-ldu', 'p-poch', 'p-rep', 'p-eco', 'p-sgv', 'p-sbv', 'p-sgb', 'p-travs', 'p-sav', 'p-vsa', 'p-vpod', 'p-voev', 'p-tcs', 'p-vcs', 'p-acs', 'p-sbk', 'p-ssv','p-gem', 'p-kdk', 'p-kkjpd', 'p-gdk', 'p-ldk', 'p-vdk', 'p-sodk', 'p-endk', 'p-fdk', 'p-edk', 'p-bpuk', 'anneepolitique', 'curiavista-de', 'swissvoteslink', 'rechtsform']]

In [272]:
df.to_csv('../data/processed/swissvotes_processed.csv', index=False)